# providers.openai

OpenAI Chat Completions via the `openai-python` SDK. Configured for Azure by default — the API shape is identical, only the auth/endpoint differs.

In [ ]:
#| default_exp providers.openai

In [ ]:
#| hide
from nbdev.showdoc import *

## The provider

`complete()` is a single round-trip: translate one OpenAI Chat Completions response into a `Turn`. The tool-dispatch loop lives in `nbdialog.core.run_completion`, not here — every chat-completion provider that supports tools would otherwise re-implement the same dance.

Construction is cheap and side-effect-free; credentials and config are resolved lazily inside `_get_client`. Endpoint and deployment have no defaults — set them via env or pass explicitly:

```python
# option 1: env vars (recommended)
# export AZURE_OPENAI_ENDPOINT=https://your-resource.cognitiveservices.azure.com
# export AZURE_OPENAI_DEPLOYMENT=your-deployment-name
# export AZURE_API_KEY=...
set_provider(OpenAIProvider())

# option 2: explicit
set_provider(OpenAIProvider(endpoint="https://...", deployment="gpt-5.4"))
```

In [ ]:
#| export
import json, os
from openai import AzureOpenAI
from nbdialog.core import Tool, Turn, ToolCall

class OpenAIProvider:
    "OpenAI Chat Completions over the `openai` SDK. `complete()` is one round-trip — driven by `run_completion` when tools are involved."
    def __init__(self,
                 deployment: str | None = None,
                 endpoint: str | None = None,
                 api_version: str = "2024-12-01-preview",
                 api_key_env: str = "AZURE_API_KEY",
                 max_completion_tokens: int = 16384):
        self.deployment = deployment or os.environ.get("AZURE_OPENAI_DEPLOYMENT")
        self.endpoint = endpoint or os.environ.get("AZURE_OPENAI_ENDPOINT")
        self.api_version, self.api_key_env = api_version, api_key_env
        self.max_completion_tokens = max_completion_tokens
        self._client = None

    def _get_client(self):
        if self._client is None:
            if not self.endpoint:
                raise RuntimeError("OpenAIProvider: no endpoint. Set $AZURE_OPENAI_ENDPOINT or pass endpoint=...")
            if not self.deployment:
                raise RuntimeError("OpenAIProvider: no deployment. Set $AZURE_OPENAI_DEPLOYMENT or pass deployment=...")
            self._client = AzureOpenAI(api_key=os.environ[self.api_key_env],
                                       azure_endpoint=self.endpoint,
                                       api_version=self.api_version)
        return self._client

    def complete(self, messages: list[dict], tools: list[Tool] = None) -> Turn:
        schemas = [t.schema for t in (tools or [])]
        kw = {"tools": schemas} if schemas else {}
        resp = self._get_client().chat.completions.create(
            model=self.deployment, messages=messages,
            max_completion_tokens=self.max_completion_tokens, **kw)
        choice = resp.choices[0]
        m = choice.message
        tcs = [ToolCall(id=tc.id, name=tc.function.name,
                        args=json.loads(tc.function.arguments))
               for tc in (m.tool_calls or [])]
        return Turn(text=m.content, tool_calls=tcs,
                    usage=resp.usage.model_dump() if resp.usage else None,
                    finish_reason=choice.finish_reason)

In [ ]:
p = OpenAIProvider(endpoint="https://demo", deployment="gpt-5.4")
p.deployment, p._client  # client is lazy — None until the first complete()

('gpt-5.4', None)

In [ ]:
#| hide
# scripted stand-in for openai.AzureOpenAI: returns queued responses, records call kwargs
from types import SimpleNamespace

def _tc_obj(call_id, name, **args):
    fn = SimpleNamespace(name=name, arguments=json.dumps(args))
    return SimpleNamespace(id=call_id, type="function", function=fn)

def _msg(content=None, tool_calls=None):
    return SimpleNamespace(content=content, tool_calls=tool_calls)

def _resp(msg, finish_reason="stop", total_tokens=10):
    usage = SimpleNamespace(model_dump=lambda: {"total_tokens": total_tokens})
    choice = SimpleNamespace(message=msg, finish_reason=finish_reason)
    return SimpleNamespace(choices=[choice], usage=usage)

class _FakeClient:
    def __init__(self, responses):
        self._responses = list(responses); self.calls = []
    @property
    def chat(self):        return self
    @property
    def completions(self): return self
    def create(self, **kw):
        self.calls.append(kw); return self._responses.pop(0)

def _provider(client, **kw):
    "OpenAIProvider wired to a fake client; explicit endpoint/deployment skip env."
    p = OpenAIProvider(endpoint="http://fake", deployment="fake", **kw)
    p._client = client
    return p

In [ ]:
#| hide
# response with content + no tool calls → Turn with text, no tool_calls, finish_reason carried through
client = _FakeClient([_resp(_msg(content="pong"), finish_reason="stop", total_tokens=12)])
prov = _provider(client)
turn = prov.complete([{"role":"user","content":"ping"}])
assert isinstance(turn, Turn)
assert turn.text == "pong"
assert turn.tool_calls == []
assert turn.finish_reason == "stop"
assert turn.usage == {"total_tokens": 12}
# no tools registered → no `tools` kwarg sent
assert "tools" not in client.calls[0]

In [ ]:
#| hide
# response with tool_calls → parsed into ToolCall list; tools schema is forwarded; truncation flagged
def _stub(**_): return ""
add_schema = {"type":"function","function":{"name":"add","parameters":{}}}
add = Tool(schema=add_schema, fn=_stub)
client = _FakeClient([_resp(_msg(tool_calls=[_tc_obj("c1","add", x=2, y=3)]),
                            finish_reason="tool_calls")])
turn = _provider(client).complete([{"role":"user","content":"2+3?"}], tools=[add])
assert turn.text is None
assert turn.finish_reason == "tool_calls"
assert len(turn.tool_calls) == 1
tc = turn.tool_calls[0]
assert (tc.id, tc.name, tc.args) == ("c1", "add", {"x":2, "y":3})
assert client.calls[0]["tools"] == [add_schema]

# length truncation should propagate so callers can surface it
client = _FakeClient([_resp(_msg(content="cut off"), finish_reason="length")])
assert _provider(client).complete([{"role":"user","content":"x"}]).finish_reason == "length"

Smoke test — only runs when credentials are present, so the doc build stays green without them.

In [ ]:
if all(os.environ.get(k) for k in ("AZURE_API_KEY", "AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_DEPLOYMENT")):
    print(OpenAIProvider().complete([{"role":"user","content":"ping"}]))

Turn(text='pong', tool_calls=[], usage={'completion_tokens': 5, 'prompt_tokens': 7, 'total_tokens': 12, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 16, 'engine_ttft_ms': 99, 'engine_ttlt_ms': 180, 'pre_inference_ms': 130, 'service_tbt_ms': 17, 'service_ttft_ms': 299, 'service_ttlt_ms': 374, 'total_duration_ms': 254, 'user_visible_ttft_ms': 169}}, finish_reason='stop')


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()